# Make synthetic clips with Wan2.1-T2V-1.3B

Use a Colab GPU runtime (T4). Run the cells in order.
First run only 1 clip and look at the time. Then make the rest.
The clips are saved in `MyDrive/compvision2_videos/<Class>/`. They are AI-generated (synthetic).
If Colab says the session crashed, use Runtime > Restart session and run all cells again from the top.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install -U diffusers transformers accelerate imageio imageio-ffmpeg ftfy sentencepiece

In [3]:
import os, time, gc, torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
from transformers import AutoTokenizer, UMT5EncoderModel

BASE = "/content/drive/MyDrive/compvision2_videos"
model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

objects = {
    "Books": "a closed book",
    "Cups": "a ceramic cup",
    "Pens": "a ballpoint pen",
    "Spoons": "a metal spoon"
}

moves = [
    "handheld camera slowly orbits around",
    "fast pan with motion blur,",
    "camera moves from top-down view to side view,",
    "slight camera shake, slowly zooming in on",
    "camera slowly moves left to right past",
    "low angle camera slowly rises above"
]

places = [
    "on a wooden table, warm lamp light",
    "on a plain white desk, bright daylight",
    "on a floor, dim evening light",
    "on a kitchen counter, mixed light",
    "on a dark table, light from a window",
    "on a bed sheet, soft morning light"
]

neg = "blurry, overexposed, low quality, distorted, deformed, extra objects, people, hands, faces, text, subtitles"

jobs = []
for i in range(4):
    for cls, name in objects.items():
        jobs.append((cls, i, f"{moves[i]} {name} {places[i]}, single object in the center, no people, photorealistic"))

print(len(jobs), "clips")

16 clips


Step 1: encode all prompts. The big text encoder goes straight to the GPU and is deleted after this cell.

In [4]:
tok = AutoTokenizer.from_pretrained(model_id, subfolder="tokenizer")
enc = UMT5EncoderModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=torch.bfloat16, device_map="cuda")

def embed(text):
    t = tok(text, padding="max_length", max_length=226, truncation=True, add_special_tokens=True,
            return_attention_mask=True, return_tensors="pt")
    ids = t.input_ids.to("cuda")
    mask = t.attention_mask.to("cuda")
    n = int(mask.gt(0).sum())
    with torch.no_grad():
        h = enc(ids, mask).last_hidden_state
    h = h[0, :n]
    h = torch.cat([h, h.new_zeros(226 - n, h.size(1))])
    return h.unsqueeze(0).cpu()

neg_emb = embed(neg)
embs = [embed(p) for _, _, p in jobs]

del enc
gc.collect()
torch.cuda.empty_cache()
print("encoded", len(embs), "prompts")

tokenizer_config.json:   0%|          | 0.00/61.8k [00:00<?, ?B/s]

tokenizer/spiece.model: reconstructing file:   0%|          |  0.00B / 4.55MB            

tokenizer/spiece.model: downloading bytes:           |  0.00B            

tokenizer/tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.8MB            

tokenizer/tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/7.08k [00:00<?, ?B/s]

... (2 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

encoded 16 prompts


Step 2: load the video model without the text encoder.

In [5]:
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=torch.bfloat16)
pipe.to("cuda")
print("ready")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B /  508MB            

vae/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

ready


Step 3: make clips. Start with `N = 1`.

In [7]:
N = 100

done = 0
for k, (cls, i, prompt) in enumerate(jobs):
    folder = BASE + "/" + cls
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/syn_{cls.lower()}_{i}.mp4"
    if os.path.exists(path):
        continue
    t = time.time()
    frames = pipe(prompt_embeds=embs[k].to("cuda"), negative_prompt_embeds=neg_emb.to("cuda"),
                  height=320, width=576, num_frames=25, num_inference_steps=15, guidance_scale=5.0).frames[0]
    export_to_video(frames, path, fps=15)
    print(path, round(time.time() - t), "sec")
    done += 1
    if done >= N:
        break

  0%|          | 0/15 [00:00<?, ?it/s]

/content/drive/MyDrive/compvision2_videos/Pens/syn_pens_0.mp4 293 sec


  0%|          | 0/15 [00:00<?, ?it/s]

/content/drive/MyDrive/compvision2_videos/Spoons/syn_spoons_0.mp4 292 sec


  0%|          | 0/15 [00:00<?, ?it/s]

/content/drive/MyDrive/compvision2_videos/Books/syn_books_1.mp4 291 sec


... (18 outputs left out)

  0%|          | 0/15 [00:00<?, ?it/s]

/content/drive/MyDrive/compvision2_videos/Pens/syn_pens_3.mp4 291 sec


  0%|          | 0/15 [00:00<?, ?it/s]

/content/drive/MyDrive/compvision2_videos/Spoons/syn_spoons_3.mp4 291 sec


Look at the first clip in Drive. If the object looks right and the time is OK, change `N = 1` to `N = 100` and run the last code cell again.
If Colab stops, restart the session, run all cells again: clips that already exist are skipped.
Open every clip and delete the bad ones (wrong or deformed object). You need about 3-4 good clips per class.